In [11]:
import pandas as pd
import joblib
import os
from imblearn.pipeline import Pipeline
from sklearn.base import clone

# Importación de modelos para Baseline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Configuración de rutas de archivos
DATA_PATH = "../data/processed/X_train_balanced_final.csv" 
PREPROC_PATH = "../models/preprocessing_pipeline.pkl"
MODELS_DIR = "../models/"

# Carga de dataset y etiquetas
df = pd.read_csv(DATA_PATH)
X_train = df.drop('Revenue', axis=1)
y_train = df['Revenue']

# Carga del pipeline de preprocesamiento del Sprint 2
preproc_pipeline = joblib.load(PREPROC_PATH)

# Reconstrucción de variables requeridas por el pipeline según documentación técnica
if 'total_paginas' not in X_train.columns:
    X_train['total_paginas'] = X_train['Administrative'] + X_train['Informational'] + X_train['ProductRelated']

if 'duracion_total' not in X_train.columns:
    X_train['duracion_total'] = X_train['Administrative_Duration'] + X_train['Informational_Duration'] + X_train['ProductRelated_Duration']

# Alineación de columnas con las características esperadas por el preprocesamiento
expected_cols = preproc_pipeline.feature_names_in_
X_train = X_train[list(expected_cols)]

# Definición de modelos con parámetros por defecto (PB-10)
models = {
    "Logistic_Regression": LogisticRegression(max_iter=1000),
    "Decision_Tree": DecisionTreeClassifier(random_state=42),
    "Random_Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "KNN": KNeighborsClassifier()
}

trained_pipelines = {}

# Entrenamiento de modelos utilizando pipelines planos para evitar anidamiento
for name, model in models.items():
    # Extracción de pasos de preprocesamiento y adición del clasificador
    steps = list(preproc_pipeline.steps)
    steps.append(('classifier', model))
    
    # Construcción y ajuste del pipeline final
    final_pipeline = Pipeline(steps)
    final_pipeline.fit(X_train, y_train)
    
    # Almacenamiento en diccionario para exportación
    trained_pipelines[name] = final_pipeline
    print(f"Modelo entrenado: {name}")

# Persistencia de modelos en formato .pkl
if not os.path.exists(MODELS_DIR):
    os.makedirs(MODELS_DIR)

for name, pipeline in trained_pipelines.items():
    save_path = os.path.join(MODELS_DIR, f"baseline_{name.lower()}.pkl")
    joblib.dump(pipeline, save_path)
    print(f"Exportado: {save_path}")

c:\Users\PerlaDorisGavilanoZa\miniforge3\envs\entorno_pucp\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\PerlaDorisGavilanoZa\miniforge3\envs\entorno_pucp\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\PerlaDorisGavilanoZa\miniforge3\envs\entorno_pucp\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpic

Modelo entrenado: Logistic_Regression
Modelo entrenado: Decision_Tree
Modelo entrenado: Random_Forest
Modelo entrenado: SVM
Modelo entrenado: KNN
Exportado: ../models/baseline_logistic_regression.pkl
Exportado: ../models/baseline_decision_tree.pkl
Exportado: ../models/baseline_random_forest.pkl
Exportado: ../models/baseline_svm.pkl
Exportado: ../models/baseline_knn.pkl
